In [1]:
import os
os.environ['USER_AGENT'] = 'BA_GPT/1.0 (contact: walrus_z@hotmail.com)'

In [2]:
import os
import sys
import yaml
import types
import pprint
import logging
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat import EVENT_LOGGER_NAME, TRACE_LOGGER_NAME
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.tools import AgentTool
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.memory.canvas import TextCanvasMemory
from autogen_ext.models.openai import OpenAIChatCompletionClient
from opentelemetry import trace, _events
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.sdk._logs import LoggerProvider
from opentelemetry.sdk._logs.export import BatchLogRecordProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk._events import EventLoggerProvider
from opentelemetry.exporter.otlp.proto.http._log_exporter import OTLPLogExporter
from opentelemetry.instrumentation.openai import OpenAIInstrumentor


# Project import paths
# Get the current directory (which is project/test)
notebook_dir = os.getcwd()

# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true"
os.environ["AZURE_SDK_TRACING_IMPLEMENTATION"] = "opentelemetry"

from agents_creator import create_agents_from_config
from rag_helpers import rag_indexer_runner
from model_loader import load_model_client
from agents_tools_helpers import selector_func

In [3]:
logging.basicConfig(level=logging.WARNING)

# For trace logging.
trace_logger = logging.getLogger(TRACE_LOGGER_NAME)
trace_logger.addHandler(logging.StreamHandler())
trace_logger.setLevel(logging.DEBUG)

# For structured message logging, such as low-level messages between agents.
event_logger = logging.getLogger(EVENT_LOGGER_NAME)
event_logger.addHandler(logging.StreamHandler())
event_logger.setLevel(logging.DEBUG)

In [4]:
# Set up resource
resource = Resource(attributes={
    "service.name": "autogen-test-agentchat"
})

# Set up tracer provider
trace.set_tracer_provider(TracerProvider(resource=resource))

# Configure OTLP exporter
otlp_exporter = OTLPSpanExporter(
    endpoint="http://localhost:4318/v1/traces"
)

# Add span processor
trace.get_tracer_provider().add_span_processor(
    BatchSpanProcessor(otlp_exporter)
)

# Set up logger provider
logger_provider = LoggerProvider(resource=resource)
logger_provider.add_log_record_processor(
    BatchLogRecordProcessor(OTLPLogExporter(endpoint="http://localhost:4318/v1/logs"))
)
_events.set_event_logger_provider(EventLoggerProvider(logger_provider))

# Enable OpenAI instrumentation
OpenAIInstrumentor().instrument()

In [5]:
rag_vector_memory = await rag_indexer_runner(lib_path + '/rag_config.yaml')

✅ All documents and URLs are already indexed.


In [6]:
# 1. Create the canvas memory and tool:
text_canvas_memory = TextCanvasMemory()
update_file_tool = text_canvas_memory.get_update_file_tool()

# 2. Register the tool as 'canvas.update_file_tool' for YAML import:
canvas = types.SimpleNamespace()
canvas.update_file_tool = update_file_tool
sys.modules['canvas'] = canvas  # This enables the tool import by your agent factory!

In [7]:
model_client = load_model_client("azure")

In [8]:
# Helpful sanity prints:
print("raw:", getattr(model_client, "_raw_config", None))
print("create_args:", getattr(model_client, "_create_args", None))

raw: {'model': 'gpt-4o', 'api_key': 'REPLACE_WITH_YOUR_API_KEY', 'azure_endpoint': 'https://224d2-m9kjdmy0-eastus2.openai.azure.com/', 'azure_deployment': 'gpt-4o', 'api_version': '2024-12-01-preview'}
create_args: {'model': 'gpt-4o'}


In [9]:
# Construct the absolute path to project/lib/agents_config.yaml user_proxy agent
AGENTS_CONFIG = os.path.abspath(os.path.join(notebook_dir, '..', 'lib', 'agents_config.yaml'))
with open(AGENTS_CONFIG, "r") as f:
        config_yaml = yaml.safe_load(f)
agent_names = list(config_yaml.keys())

user, agents = await create_agents_from_config(
    AGENTS_CONFIG,
    agent_names=agent_names,
    model_client=model_client,
    vector_memory=rag_vector_memory,
    canvas_memory=text_canvas_memory,
    input_func=input
)

In [10]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    pprint.pprint(vars(agent))




--- Agent: planning_agent ---
{'_description': 'The orchestrator agent responsible for breaking down a new '
                 'task into subtasks, assigning them to the right agents, and '
                 'ensuring the workflow progresses smoothly until completion.',
 '_handoff_tools': [],
 '_handoffs': {},
 '_is_running': False,
 '_max_tool_iterations': 1,
 '_memory': None,
 '_metadata': {},
 '_model_client': <autogen_ext.models.openai._openai_client.AzureOpenAIChatCompletionClient object at 0x7f35c2551be0>,
 '_model_client_stream': True,
 '_model_context': <autogen_core.model_context._unbounded_chat_completion_context.UnboundedChatCompletionContext object at 0x7f35c2552660>,
 '_name': 'planning_agent',
 '_output_content_type': None,
 '_output_content_type_format': None,
 '_reflect_on_tool_use': False,
 '_structured_message_factory': None,
 '_system_messages': [SystemMessage(content='You are the PLANNING AGENT, the coordinator for this multi-agent system.\n\n**YOUR MISSION**\n  Brea

In [11]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    print("Tools:")
    for tool in agent._tools:
        print(f"Tool Name: {tool.name}")
        print(f"Description: {tool.description}\n")



--- Agent: planning_agent ---
Tools:

--- Agent: interviewer_agent ---
Tools:
Tool Name: save_conversation_to_jsonl
Description: 
Saves either:
1) A Q&A pair (interviewer + user) in one atomic JSONL write, OR
2) A generic list of records.



--- Agent: summarizer_agent ---
Tools:

--- Agent: functional_spec_writer_agent ---
Tools:
Tool Name: write_md_file_tool
Description: 
Writes the functional specification to a Markdown file.


Tool Name: use_case_agent
Description: An agent dedicated to generating Use Cases and User Stories from the project information.

Tool Name: glossary_agent
Description: An agent dedicated to building a Glossary of terms from the gathered requirements and project information.


--- Agent: functional_spec_checker_agent ---
Tools:
Tool Name: read_md_file_tool
Description: 
Reads the entire content of a Markdown file and returns it as a string.

Args:
    file_path (str): Optional path to the Markdown spec (defaults to DEFAULT_MD_SPEC_PATH).

Returns:
    str: T

In [12]:
selector_prompt = """
Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""

In [13]:
# Order agents (workflow order), 
agent_order = [
    "planning_agent",
    "interviewer_agent",
    "summarizer_agent",
    "functional_spec_writer_agent",
    "functional_spec_checker_agent",
    "diagram_creator_agent",
    "mermaid_code_reviewer_agent",
    "terminate_agent"
]
member_agents = [agents[name] for name in agent_order if name in agents] + [user]

In [14]:
termination = MaxMessageTermination(100000000) | TextMentionTermination("TERMINATE")

# --- Use SelectorGroupChat ---
team = SelectorGroupChat(
    member_agents,
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=False,
    model_client_streaming=False,
    selector_func=selector_func,
)

In [15]:
task = "Interview a customer to collect requirements and scope of a new product then write the functional specifications"

In [16]:
runtime = SingleThreadedAgentRuntime(
            tracer_provider=trace.NoOpTracerProvider(),  # Disable telemetry for runtime.
        )
runtime.start()

In [ ]:
try:
    await Console(team.run_stream(task=task))
except Exception as e:
    print(f"Error during team run_stream: {e}")

await runtime.stop()

try:
    await model_client.close()
except Exception as e:
    print(f"Error closing model_client: {e}")

---------- TextMessage (user) ----------
Interview a customer to collect requirements and scope of a new product then write the functional specifications
---------- MemoryQueryEvent (interviewer_agent) ----------
[MemoryContent(content='fosters a sense of ownership and satisfaction among all parties involved, leading to more effective and successful project outcomes.Our product Requiment requirements gathering software, guides you through prepared questions to determine a full and detailed requirement specification and project scope based on outcomes. This application acts as a virtual business analyst, guiding users through the requirements process, whilst providing insight and recommendations to the user based on the type of', mime_type='MemoryMimeType.TEXT', metadata={'chunk_index': 47, 'source': 'https://www.requiment.com/the-most-asked-questions-about-requirements-gathering/', 'mime_type': 'MemoryMimeType.TEXT', 'score': 0.5941268801689148, 'id': 'd43b373f-f1ce-49e4-b5bf-c67e3fb22

id='e7355f30-5b97-437d-b077-680803ba0ec0' source='interviewer_agent' models_usage=RequestUsage(prompt_tokens=0, completion_tokens=0) metadata={} created_at=datetime.datetime(2025, 9, 13, 20, 33, 30, 611719, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='call_9lx7KlJOLwr6LoeOczkdaoX6', arguments='{"speaker_or_records":[{"speaker":"interviewer","message":"QUESTION: Can you describe the purpose and main goals of the new product you have in mind?","subject":"Product Overview","q_index":1},{"speaker":"user","message":"The scope of the project is to design and develop an online ecommerce website for apparels.Customers will be able to use the frontend website for searching the apparels and place theorder by making online payment of the order in advance. Web based admin panel/ backend willhelp admin user i.e. business owner to manage the products, categories, prices and ordersplaced by the customers.","subject":"Product Overview","q_index":1}],"file_path":"../conversation/interview.js

---------- ToolCallRequestEvent (interviewer_agent) ----------
[FunctionCall(id='call_9lx7KlJOLwr6LoeOczkdaoX6', arguments='{"speaker_or_records":[{"speaker":"interviewer","message":"QUESTION: Can you describe the purpose and main goals of the new product you have in mind?","subject":"Product Overview","q_index":1},{"speaker":"user","message":"The scope of the project is to design and develop an online ecommerce website for apparels.Customers will be able to use the frontend website for searching the apparels and place theorder by making online payment of the order in advance. Web based admin panel/ backend willhelp admin user i.e. business owner to manage the products, categories, prices and ordersplaced by the customers.","subject":"Product Overview","q_index":1}],"file_path":"../conversation/interview.jsonl","session_id":"sess-20230316124530"}', name='save_conversation_to_jsonl')]


id='1f651557-ff81-4836-9d84-5bef6850a9df' source='interviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 13, 20, 33, 30, 615034, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='Saved 2 record(s) to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='call_9lx7KlJOLwr6LoeOczkdaoX6', is_error=False)] type='ToolCallExecutionEvent'
id='1f651557-ff81-4836-9d84-5bef6850a9df' source='interviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 13, 20, 33, 30, 615034, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='Saved 2 record(s) to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='call_9lx7KlJOLwr6LoeOczkdaoX6', is_error=False)] type='ToolCallExecutionEvent'
DEBUG:autogen_agentchat.events:id='1f651557-ff81-4836-9d84-5bef6850a9df' source='interviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 13, 20, 33

---------- ToolCallExecutionEvent (interviewer_agent) ----------
[FunctionExecutionResult(content='Saved 2 record(s) to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='call_9lx7KlJOLwr6LoeOczkdaoX6', is_error=False)]
---------- ModelClientStreamingChunkEvent (interviewer_agent) ----------
QUESTION: What key features and capabilities are you considering to ensure the product meets its goals effectively?
---------- TextMessage (user) ----------
Objectives: Enhancing client’s offline business of apparels by transforming it into online ecommerce website., Making online ecommerce platform for buyers to buy apparels from website., Customers/ buyers will be able to search for the various kinds of apparels for e.g. Shirts. Jeans etc, and will be able to order it from website by making online payment of products. Business owner will be able to manage his products by category and prices with different sizes & colors. Owner will be able to deliver the items to the bu

id='17b17024-9451-422a-b720-774856bc3bf6' source='interviewer_agent' models_usage=RequestUsage(prompt_tokens=0, completion_tokens=0) metadata={} created_at=datetime.datetime(2025, 9, 13, 20, 33, 58, 464504, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='call_SwDxZxezQoy25ZxNlJnwBsYo', arguments='{"speaker_or_records":[{"speaker":"interviewer","message":"QUESTION: What key features and capabilities are you considering to ensure the product meets its goals effectively?","subject":"Product Overview","q_index":2},{"speaker":"user","message":"Objectives: Enhancing client’s offline business of apparels by transforming it into online ecommerce website., Making online ecommerce platform for buyers to buy apparels from website., Customers/ buyers will be able to search for the various kinds of apparels for e.g. Shirts. Jeans etc, and will be able to order it from website by making online payment of products. Business owner will be able to manage his products by category and prices with

... (55 outputs left out)

id='b7c608d7-2b9b-4bef-894c-16239ea839ea' source='interviewer_agent' models_usage=RequestUsage(prompt_tokens=0, completion_tokens=0) metadata={} created_at=datetime.datetime(2025, 9, 13, 20, 40, 4, 67922, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='call_zIDf2Y2WIKVUv6ZOeoVub4Wx', arguments='{"speaker_or_records":[{"speaker":"interviewer","message":"QUESTION: Are there any high-priority risks identified so far that need immediate mitigation plans?","subject":"Dependencies & Risks","q_index":2},{"speaker":"user","message":"Risks: Lack of training to the employees regarding use of system. Issues:Auditor approval of our new process & system, Senior management buy-off on the, approach, Funding for this initiative, Developer resources","subject":"Dependencies & Risks","q_index":2}],"file_path":"../conversation/interview.jsonl","session_id":"sess-20230316124530"}', name='save_conversation_to_jsonl')] type='ToolCallRequestEvent'
id='b7c608d7-2b9b-4bef-894c-16239ea839ea' source='int

---------- ToolCallRequestEvent (interviewer_agent) ----------
[FunctionCall(id='call_zIDf2Y2WIKVUv6ZOeoVub4Wx', arguments='{"speaker_or_records":[{"speaker":"interviewer","message":"QUESTION: Are there any high-priority risks identified so far that need immediate mitigation plans?","subject":"Dependencies & Risks","q_index":2},{"speaker":"user","message":"Risks: Lack of training to the employees regarding use of system. Issues:Auditor approval of our new process & system, Senior management buy-off on the, approach, Funding for this initiative, Developer resources","subject":"Dependencies & Risks","q_index":2}],"file_path":"../conversation/interview.jsonl","session_id":"sess-20230316124530"}', name='save_conversation_to_jsonl')]


id='4948e1e1-911c-4ade-b764-80a4e0a99e65' source='interviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 13, 20, 40, 4, 70648, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='Saved 2 record(s) to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='call_zIDf2Y2WIKVUv6ZOeoVub4Wx', is_error=False)] type='ToolCallExecutionEvent'
id='4948e1e1-911c-4ade-b764-80a4e0a99e65' source='interviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 13, 20, 40, 4, 70648, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='Saved 2 record(s) to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='call_zIDf2Y2WIKVUv6ZOeoVub4Wx', is_error=False)] type='ToolCallExecutionEvent'
DEBUG:autogen_agentchat.events:id='4948e1e1-911c-4ade-b764-80a4e0a99e65' source='interviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 13, 20, 40, 4,

---------- ToolCallExecutionEvent (interviewer_agent) ----------
[FunctionExecutionResult(content='Saved 2 record(s) to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='call_zIDf2Y2WIKVUv6ZOeoVub4Wx', is_error=False)]
---------- ModelClientStreamingChunkEvent (interviewer_agent) ----------
Product Overview  asked=2, status=done
KPI & Success Metrics  asked=2, status=done
Scope In and Out  asked=2, status=done
Non-Functional Requirements  asked=2, status=done
User Roles & Personas  asked=2, status=done
Integrations & External Systems  asked=2, status=done
Constraints & Assumptions  asked=2, status=done
Dependencies & Risks  asked=2, status=done

STATUS:
  Product Overview  asked=2, status=done
  KPI & Success Metrics  asked=2, status=done
  Scope In and Out  asked=2, status=done
  Non-Functional Requirements  asked=2, status=done
  User Roles & Personas  asked=2, status=done
  Integrations & External Systems  asked=2, status=done
  Constraints & Assumptions 